# OpenCLIP Binary Classification

## Setup and Imports

In [1]:
import torch
import os

from PIL import Image
import open_clip
import numpy as np
import pandas as pd
import joblib

from nazi_symbols_classification.training.data_preparation import get_image_paths
from nazi_symbols_classification.training.evaluation import get_top1_evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

/mnt/data/nazi-symbols-classification/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Load and Prepare Data

In [2]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-detection", ("train", "test", "val"))

In [3]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/val')]

In [4]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

## Load OpenCLIP Model

In [5]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.to("cuda")

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

## Classify Images

Define a function to classify images using the OpenCLIP model. The function takes an image path and a list of prompts, encodes the image and text, and computes the probabilities for each prompt.

In [6]:
def classify_image(image_path, prompts):
    text = tokenizer(prompts)
    with torch.no_grad(), torch.autocast("cuda"):
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image.cuda())
        text_features = model.encode_text(text.cuda())  # type: ignore
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        return {k:round(v.item(), 3) for k, v in dict(zip(prompts, text_probs[0].cpu())).items()}

In [7]:
prompts = {
    "A black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism.": "nazi",
    "The British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.": "nazi",
    "A broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.": "nazi",
    "Historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.": "nazi",
    "Images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.": "nazi",
    "Images of the Judenstern, the yellow Star of David badge used during the Holocaust, often featuring the word 'Jude' in black lettering in the center.": "nazi",
    "Images featuring the 'Happy Merchant' meme, a stereotypical representation of a smiling, hook-nosed Jewish figure used in online racist and antisemitic contexts.": "nazi",
    "Imagery of neo-Nazi groups featuring hate symbols like swastikas, Black Sun, Siegrune, or Celtic Cross on flags, banners, clothing, or graffiti, often seen at rallies, protests, or in propaganda materials promoting white supremacy and far-right ideology.": "nazi",
    "A single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "nazi",
    "A skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "nazi",
    "a brownshirt (SA) symbol present in this image": "nazi",
    "A black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "nazi",
    "The Wolfsangel symbol, resembling a hook-like rune, used by Nazi groups and German military units during World War II.": "nazi",
    "An image containing no Nazi-related content.": "non-nazi"
}

In [9]:
prompts.values()

dict_values(['nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'nazi', 'non-nazi'])

In [12]:
tmp_result = {"nazi": 0.0, "non-nazi": 0.0}
classify_result = classify_image(test_images[0], prompts.keys())
for k, v in classify_result.items():
    tmp_result[prompts[k]] += v
print(tmp_result)
sorted(tmp_result.items(), key=lambda x: x[1], reverse=True)[0][0]

{'nazi': 0.8839999999999999, 'non-nazi': 0.116}


'nazi'

## Classify All Test Images

In [8]:
%%time

probs = []

for image_path in test_images:
    tmp_result = {"nazi": 0.0, "non-nazi": 0.0}
    classify_result = classify_image(image_path, prompts.keys())
    for k, v in classify_result.items():
        tmp_result[prompts[k]] += v
    # result.append(sorted(tmp_result.items(), key=lambda x: x[1], reverse=True)[0][0])
    probs.append(tmp_result["nazi"])

CPU times: user 1h 9s, sys: 4.6 s, total: 1h 14s
Wall time: 4min 42s


## Save Results

In [24]:
result = dict(y_true=[int(label == "nazi-symbol") for label in y_test], 
              probs=probs, 
              outputs=[int(prob > 0.5) for prob in probs])

joblib.dump(result, "openclip-output/open_clip_result_binary.joblib")

['openclip-output/open_clip_result_v2-binary.joblib']

In [11]:
len(probs)

15018

## Evaluate Results 

In [19]:
print(classification_report(["nazi" if label == "nazi-symbol" else "non-nazi" for label in y_test], result["outputs"], digits=3))

              precision    recall  f1-score   support

        nazi      0.011     0.649     0.021       205
    non-nazi      0.972     0.171     0.291     14813

    accuracy                          0.178     15018
   macro avg      0.492     0.410     0.156     15018
weighted avg      0.959     0.178     0.288     15018



In [22]:
y_test_tmp = [int(label == "nazi-symbol") for label in y_test]
roc_auc_score(y_test_tmp, probs), accuracy_score(y_test_tmp, [int(prob > 0.5) for prob in probs])

(0.4235686188631278, 0.17778665601278465)